# AlphaLOB Phase 2 — Notebook 05: Institutional-Grade Walk-Forward Backtest

**Input:** `/content/lob_features.parquet`, `/content/lobster_transformer.onnx`, `/content/regime_hmm.pkl`

**Output:** Metrics table + equity curve PNG + walk-forward comparison chart

## Walk-Forward Structure (3 OOS Windows)
```
Window 1: Train [0%→50%]  Test [50%→60%]
Window 2: Train [0%→60%]  Test [60%→70%]
Window 3: Train [0%→70%]  Test [70%→80%]
Final OOS: [80%→100%]  ← NEVER TOUCHED until final evaluation
```

## Interview Key Points
- **Walk-forward** = no cherry-picking. Mean Sharpe across 3 windows, not the best one.
- **Square-root market impact** = realistic execution cost model.
- **Break-even transaction cost** = answers "at what cost does your strategy stop working?"
- **Kelly position sizing** = mathematically optimal bet-size given edge.

---

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# Cell 1: Install dependencies
!pip install polars pyarrow onnxruntime hmmlearn joblib matplotlib --quiet
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Imports
import numpy as np
import polars as pl
import onnxruntime as ort
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
warnings.filterwarnings('ignore')

PARQUET_IN = '/content/drive/MyDrive/AlphaLOB/lob_features.parquet'
ONNX_PATH  = '/content/drive/MyDrive/AlphaLOB/lobster_transformer.onnx'
HMM_PATH   = '/content/drive/MyDrive/AlphaLOB/regime_hmm.pkl'

# Walk-forward splits (as fractions of total data)
WINDOWS = [
    {'train_end': 0.50, 'test_start': 0.50, 'test_end': 0.60},
    {'train_end': 0.60, 'test_start': 0.60, 'test_end': 0.70},
    {'train_end': 0.70, 'test_start': 0.70, 'test_end': 0.80},
]
FINAL_OOS_START = 0.80  # NEVER TOUCH until final evaluation

# Strategy parameters
LONG_THRESHOLD   = 0.60   # Enter LONG if dir_30s_prob(UP) > 0.60
SHORT_THRESHOLD  = 0.40   # Enter SHORT if dir_30s_prob(UP) < 0.40
KELLY_CAP        = 0.25   # Maximum position size (25% of capital)
STOP_LOSS        = -0.005 # -0.5% per trade stop
INITIAL_CAPITAL  = 100_000.0

# Cost model
SLIPPAGE_MIN_BPS = 0.5
SLIPPAGE_MAX_BPS = 2.0
DAILY_VOL_FRAC   = 0.01   # assume we trade 1% of ADV

print('✅ Configuration set')

In [ ]:
# Cell 3: Load all data and models

print('Loading data...')
df = pl.read_parquet(PARQUET_IN)
df = df.fill_null(0.0).fill_nan(0.0)
print(f'  {len(df):,} rows loaded')

print('Loading ONNX model...')
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
print(f'  ONNX inputs:  {[i.name for i in sess.get_inputs()]}')
print(f'  ONNX outputs: {[o.name for o in sess.get_outputs()]}')

print('Loading RegimeHMM...')
hmm_model = joblib.load(HMM_PATH)
regime_names = hmm_model.regime_names
print(f'  Regime labels: {regime_names}')

n_total = len(df)

In [ ]:
# Cell 4: Build LOB input tensor for ONNX (same as Notebook 03 LOBDataset)

def build_lob_tensor(df_slice: pl.DataFrame) -> np.ndarray:
    """Build (n, 10, 4) float32 array for ONNX inference."""
    n = len(df_slice)
    N_LEVELS = 10
    X = np.zeros((n, N_LEVELS, 4), dtype=np.float32)
    mid = df_slice['mid_price'].to_numpy()
    wofi = df_slice['wofi_z'].to_numpy()
    kyle = df_slice['kyle_lambda_z'].to_numpy()

    for lvl in range(N_LEVELS):
        bp = df_slice[f'bid_price_{lvl}'].to_numpy()
        ap = df_slice[f'ask_price_{lvl}'].to_numpy()
        bv = df_slice[f'bid_vol_{lvl}'].to_numpy()
        av = df_slice[f'ask_vol_{lvl}'].to_numpy()
        price_dist = ((bp + ap) / 2 - mid) / (mid + 1e-9)
        vol_total  = np.log1p(bv + av)
        X[:, lvl, 0] = price_dist
        X[:, lvl, 1] = vol_total
        X[:, lvl, 2] = wofi
        X[:, lvl, 3] = kyle
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X


def run_inference_batch(X: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    """Run ONNX inference in batches. Returns dir_30s UP probability."""
    all_probs = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        outputs = sess.run(['dir_30s'], {'lob_snapshot': batch})
        # outputs[0] shape: (batch, 2) — [prob_DOWN, prob_UP]
        all_probs.append(outputs[0][:, 1])  # prob_UP
    return np.concatenate(all_probs)


def get_regimes(df_slice: pl.DataFrame) -> np.ndarray:
    """Predict HMM regimes for a data slice."""
    X_hmm = np.column_stack([
        df_slice['realized_vol'].to_numpy(),
        df_slice['autocorrelation'].to_numpy()
    ]).astype(np.float64)
    X_hmm = np.nan_to_num(X_hmm)
    states = hmm_model.predict(X_hmm)
    return np.array([regime_names[s] for s in states])


print('✅ Helper functions defined')

In [ ]:
# Cell 5: Market impact model (square-root model)
# market_impact = σ · √(Q / ADV) · sign(Q)
# σ = daily volatility, Q = order size, ADV from training window (point-in-time)

def compute_sqrt_market_impact(Q_frac: float, daily_vol: float, adv_frac: float = DAILY_VOL_FRAC) -> float:
    """
    Square-root market impact model.
    Q_frac: fraction of ADV being traded (e.g., 0.01 = 1%)
    daily_vol: realized daily volatility (e.g., 0.02 = 2%)
    Returns: impact in basis points
    """
    # η is the market impact coefficient (typically 0.5–1.0 for liquid crypto)
    eta = 0.5
    impact_pct = eta * daily_vol * np.sqrt(Q_frac)
    return impact_pct * 10_000  # convert to bps


def kelly_fraction(prob_up: float, win_frac: float = 1.0, loss_frac: float = 1.0) -> float:
    """
    Kelly criterion for binary bet:
    f* = (p·b - q) / b   where b = win_frac/loss_frac
    p = prob_up, q = 1-p
    """
    p = prob_up; q = 1 - p; b = win_frac / loss_frac
    f = (p * b - q) / b
    return np.clip(f, 0.0, KELLY_CAP)  # cap at 25%, never negative


print('✅ Market impact and Kelly fraction functions defined')

In [ ]:
# Cell 6: Risk metrics computation (all in pure NumPy/Polars — no quantstats needed)

def compute_metrics(returns: np.ndarray, trades: list) -> dict:
    """
    Compute all required quant metrics from a series of trade P&L.
    returns: array of per-trade returns (as fractions, e.g., 0.001 = 0.1%)
    trades: list of dicts with 'pnl', 'entry', 'exit' etc.
    """
    if len(returns) == 0:
        return {k: np.nan for k in ['sharpe_ratio','sortino_ratio','calmar_ratio',
                                     'omega_ratio','max_drawdown','total_return',
                                     'win_rate','profit_factor','avg_trade_pnl',
                                     'break_even_bps','total_trades','ann_return']}

    # Annualization factor (trades_per_year)
    TICKS_PER_YEAR = 10 * 60 * 60 * 24 * 365  # 10 ticks/sec
    trades_per_year = max(len(returns), 1)

    total_return = (1 + returns).prod() - 1
    mean_ret     = returns.mean()
    std_ret      = returns.std(ddof=1) if len(returns) > 1 else 1e-9

    # Annualized return
    ann_return = (1 + total_return) ** (252 / max(len(returns) / trades_per_year * 252, 1)) - 1

    # Sharpe Ratio
    rf_per_trade = 0.0  # assume zero risk-free rate
    sharpe = (mean_ret - rf_per_trade) / std_ret * np.sqrt(trades_per_year)

    # Sortino Ratio (only penalizes downside vol)
    downside = returns[returns < 0]
    downside_std = downside.std(ddof=1) if len(downside) > 1 else 1e-9
    sortino = mean_ret / downside_std * np.sqrt(trades_per_year)

    # Maximum Drawdown
    cum_rets = (1 + returns).cumprod()
    rolling_max = np.maximum.accumulate(cum_rets)
    drawdowns = (cum_rets - rolling_max) / rolling_max
    max_dd = abs(drawdowns.min())

    # Drawdown duration
    in_dd = drawdowns < 0
    dd_lengths = []
    count = 0
    for x in in_dd:
        if x: count += 1
        else:
            if count > 0: dd_lengths.append(count)
            count = 0
    max_dd_dur = max(dd_lengths) if dd_lengths else 0

    # Calmar Ratio
    calmar = ann_return / max_dd if max_dd > 0 else np.nan

    # Omega Ratio (probability-weighted gain/loss)
    gains  = returns[returns >= 0].sum()
    losses = abs(returns[returns < 0].sum())
    omega  = gains / losses if losses > 0 else np.nan

    # Win rate and profit factor
    win_rate = (returns > 0).mean()
    gross_profit = returns[returns > 0].sum()
    gross_loss   = abs(returns[returns < 0].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.nan

    # Break-even transaction cost
    # At this bps level, expected return per trade = 0
    # break_even = mean_return_per_trade / 1 (binary payoff) * 10000 (bps)
    break_even_bps = mean_ret * 10_000  # bps per trade

    return {
        'sharpe_ratio':    sharpe,
        'sortino_ratio':   sortino,
        'calmar_ratio':    calmar,
        'omega_ratio':     omega,
        'max_drawdown':    max_dd,
        'max_dd_duration': max_dd_dur,
        'total_return':    total_return,
        'ann_return':      ann_return,
        'win_rate':        win_rate,
        'profit_factor':   profit_factor,
        'avg_trade_pnl':   mean_ret,
        'break_even_bps':  break_even_bps,
        'total_trades':    len(returns),
    }

print('✅ Metrics function defined')

In [ ]:
# Cell 7: Core backtesting engine (vectorized with Polars)

def run_backtest_on_window(df_test_slice: pl.DataFrame,
                            daily_vol: float,
                            window_name: str) -> dict:
    """
    Run full backtest on one OOS window.
    Uses ONNX predictions + HMM regimes + Kelly sizing + square-root impact.
    """
    print(f'  Running backtest: {window_name} ({len(df_test_slice):,} ticks)...')
    t0 = time.time()

    # 1. Build model inputs
    X = build_lob_tensor(df_test_slice)

    # 2. Get ONNX predictions
    prob_up = run_inference_batch(X)  # (n,) float — probability of UP

    # 3. Get regime labels
    regimes = get_regimes(df_test_slice)  # (n,) string array

    # 4. Get mid prices
    mid = df_test_slice['mid_price'].to_numpy()
    HORIZON_TICKS = 300  # 30s at 10 ticks/sec

    # 5. Simulate trade-by-trade
    trade_returns = []
    directional_correct = 0
    total_predictions   = 0

    rng = np.random.default_rng(seed=42)
    i = 0
    while i < len(mid) - HORIZON_TICKS:
        p_up = prob_up[i]
        regime = regimes[i]

        # Signal generation
        if regime == 'VOLATILE':
            i += 1
            continue  # No trading in volatile regime

        if p_up > LONG_THRESHOLD:
            direction = 1  # LONG
        elif p_up < SHORT_THRESHOLD:
            direction = -1  # SHORT
        else:
            i += 1
            continue  # No trade

        # Kelly position sizing
        f_kelly = kelly_fraction(p_up if direction == 1 else (1 - p_up))
        if f_kelly < 0.01:
            i += 1
            continue  # Edge too small

        # Entry/exit prices
        entry_price = mid[i]
        exit_price  = mid[i + HORIZON_TICKS]

        # Transaction costs
        slippage_bps = rng.uniform(SLIPPAGE_MIN_BPS, SLIPPAGE_MAX_BPS)
        impact_bps   = compute_sqrt_market_impact(DAILY_VOL_FRAC, daily_vol)
        total_cost_bps = slippage_bps + impact_bps
        total_cost_pct = total_cost_bps / 10_000

        # Raw return (as fraction)
        raw_ret = direction * (exit_price - entry_price) / entry_price

        # Apply stop-loss
        if raw_ret < STOP_LOSS:
            raw_ret = STOP_LOSS

        # Net return after costs (costs apply on both entry and exit)
        net_ret = f_kelly * (raw_ret - 2 * total_cost_pct)

        trade_returns.append(net_ret)

        # Directional accuracy tracking
        actual_direction = 1 if exit_price > entry_price else -1
        if direction == actual_direction:
            directional_correct += 1
        total_predictions += 1

        # Advance by HORIZON_TICKS to avoid overlapping trades
        i += HORIZON_TICKS

    elapsed = time.time() - t0
    trade_returns = np.array(trade_returns)
    dir_accuracy = directional_correct / max(total_predictions, 1)

    metrics = compute_metrics(trade_returns, [])
    metrics['directional_acc_30s'] = dir_accuracy
    metrics['window'] = window_name
    metrics['elapsed_s'] = elapsed

    print(f'    Done in {elapsed:.1f}s | Trades: {len(trade_returns)} | '
          f'Sharpe: {metrics["sharpe_ratio"]:.2f} | '
          f'Acc: {dir_accuracy*100:.1f}%')

    return metrics, trade_returns

print('✅ Backtest engine defined')

In [ ]:
# Cell 8: Run all 3 walk-forward windows

print('=' * 60)
print('  WALK-FORWARD BACKTEST — 3 OOS WINDOWS')
print('=' * 60)

all_metrics = []
all_returns = []

for w_idx, window in enumerate(WINDOWS):
    window_name = f'Window {w_idx+1}'

    # Get test slice (chronological)
    test_start_idx = int(n_total * window['test_start'])
    test_end_idx   = int(n_total * window['test_end'])
    train_end_idx  = int(n_total * window['train_end'])

    df_test  = df.slice(test_start_idx, test_end_idx - test_start_idx)
    df_train = df.slice(0, train_end_idx)

    # Compute daily volatility on TRAINING data only (point-in-time — no look-ahead)
    train_mid    = df_train['mid_price'].to_numpy()
    train_rets   = np.diff(np.log(train_mid))
    daily_vol    = train_rets.std() * np.sqrt(10 * 60 * 60 * 24)  # annualize

    metrics, returns = run_backtest_on_window(df_test, daily_vol, window_name)
    all_metrics.append(metrics)
    all_returns.append(returns)

print('\n✅ All 3 walk-forward windows complete')

In [ ]:
# Cell 9: Print full metrics table

print('\n' + '=' * 70)
print('  WALK-FORWARD BACKTEST RESULTS')
print('=' * 70)

METRIC_KEYS = [
    ('sharpe_ratio',       'Sharpe Ratio',          '2.3',   '{:.3f}'),
    ('sortino_ratio',      'Sortino Ratio',          '3.1',   '{:.3f}'),
    ('calmar_ratio',       'Calmar Ratio',           '>1.0',  '{:.3f}'),
    ('omega_ratio',        'Omega Ratio',            '>1.0',  '{:.3f}'),
    ('max_drawdown',       'Max Drawdown',           '8.3%',  '{:.2%}'),
    ('max_dd_duration',    'Max DD Duration (trades)','—',    '{:.0f}'),
    ('total_return',       'Total Return',           '—',     '{:.2%}'),
    ('ann_return',         'Annualized Return',      '—',     '{:.2%}'),
    ('win_rate',           'Win Rate',               '—',     '{:.2%}'),
    ('profit_factor',      'Profit Factor',          '>1.5',  '{:.3f}'),
    ('avg_trade_pnl',      'Avg Trade P&L',          '—',     '{:.6f}'),
    ('break_even_bps',     'Break-Even Cost (bps)',  '8.2',   '{:.2f}'),
    ('total_trades',       'Total Trades',           '—',     '{:.0f}'),
    ('directional_acc_30s','30s Dir. Accuracy',      '58.2%', '{:.2%}'),
]

# Header
header = f'{"Metric":<32} {"Target":>8}'
for m in all_metrics:
    header += f'  {m["window"]:>12}'
header += f'  {"MEAN":>12}'
print(header)
print('-' * 80)

# Rows
for key, label, target, fmt in METRIC_KEYS:
    row = f'{label:<32} {target:>8}'
    vals = []
    for m in all_metrics:
        v = m.get(key, np.nan)
        row += f'  {fmt.format(v):>12}'
        vals.append(v)
    mean_v = np.nanmean(vals)
    row += f'  {fmt.format(mean_v):>12}'
    print(row)

print('=' * 70)
mean_sharpe = np.mean([m['sharpe_ratio'] for m in all_metrics])
mean_acc    = np.mean([m['directional_acc_30s'] for m in all_metrics])
mean_be     = np.mean([m['break_even_bps'] for m in all_metrics])
print(f'\n  ★ Mean Sharpe (3 OOS windows): {mean_sharpe:.3f}  (target: 2.3)')
print(f'  ★ Mean 30s Accuracy:           {mean_acc*100:.2f}%   (target: 58.2%)')
print(f'  ★ Mean Break-even Cost:        {mean_be:.2f} bps      (target: 8.2 bps)')

In [ ]:
# Cell 10: Plot equity curves and walk-forward comparison

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.3)
fig.suptitle('AlphaLOB Walk-Forward Backtest Results', fontsize=14, fontweight='bold')

COLORS = ['#2196F3', '#4CAF50', '#FF9800']

# Top row: equity curves for each window
for w_idx, (returns, metrics) in enumerate(zip(all_returns, all_metrics)):
    ax = fig.add_subplot(gs[0, w_idx])
    if len(returns) > 0:
        equity = INITIAL_CAPITAL * (1 + returns).cumprod()
        ax.plot(equity, color=COLORS[w_idx], linewidth=1.5)
        ax.axhline(INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5)
        ax.fill_between(range(len(equity)), INITIAL_CAPITAL, equity,
                        alpha=0.2, color=COLORS[w_idx])
    ax.set_title(f'{metrics["window"]}\nSharpe={metrics["sharpe_ratio"]:.2f} | '
                 f'DD={metrics["max_drawdown"]:.1%}', fontsize=10)
    ax.set_xlabel('Trades')
    ax.set_ylabel('Capital ($)')
    ax.grid(alpha=0.3)

# Bottom left: Sharpe comparison across windows
ax_sharpe = fig.add_subplot(gs[1, 0])
sharpes   = [m['sharpe_ratio'] for m in all_metrics]
bars = ax_sharpe.bar([m['window'] for m in all_metrics], sharpes, color=COLORS)
ax_sharpe.axhline(2.3, color='red', linestyle='--', alpha=0.7, label='Target (2.3)')
ax_sharpe.axhline(np.mean(sharpes), color='gold', linestyle='-', alpha=0.8,
                   label=f'Mean ({np.mean(sharpes):.2f})')
ax_sharpe.set_title('Sharpe Ratio by Window')
ax_sharpe.legend(fontsize=8)
ax_sharpe.grid(alpha=0.3, axis='y')

# Bottom middle: Directional accuracy comparison
ax_acc = fig.add_subplot(gs[1, 1])
accs   = [m['directional_acc_30s']*100 for m in all_metrics]
ax_acc.bar([m['window'] for m in all_metrics], accs, color=COLORS)
ax_acc.axhline(50,   color='red',    linestyle='--', alpha=0.7, label='Random (50%)')
ax_acc.axhline(58.2, color='orange', linestyle='--', alpha=0.7, label='Target (58.2%)')
ax_acc.set_title('30s Directional Accuracy (%)')
ax_acc.legend(fontsize=8)
ax_acc.grid(alpha=0.3, axis='y')

# Bottom right: Break-even cost vs actual slippage
ax_cost = fig.add_subplot(gs[1, 2])
be_costs = [m['break_even_bps'] for m in all_metrics]
ax_cost.bar([m['window'] for m in all_metrics], be_costs, color=COLORS)
ax_cost.axhline(SLIPPAGE_MAX_BPS, color='red', linestyle='--',
                label=f'Max slippage ({SLIPPAGE_MAX_BPS} bps)')
ax_cost.axhline(8.2, color='gold', linestyle='--', alpha=0.7, label='Target (8.2 bps)')
ax_cost.set_title('Break-Even Transaction Cost (bps)')
ax_cost.legend(fontsize=8)
ax_cost.grid(alpha=0.3, axis='y')

plt.savefig('/content/walkforward_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Walk-forward results saved to /content/walkforward_results.png')

In [ ]:
# Cell 11: Final OOS evaluation (the NEVER-TOUCHED 80-100% slice)
# Only run this once, after you are satisfied with the walk-forward results
# Running this multiple times is data snooping!

print('=== FINAL OUT-OF-SAMPLE EVALUATION (80-100%) ===')
print('⚠️  This data was never touched during model selection.')
print('   Running this cell a second time constitutes data snooping!')

final_oos_idx = int(n_total * FINAL_OOS_START)
df_final_oos  = df.slice(final_oos_idx, n_total - final_oos_idx)

# Use daily vol from 0-80% of data (point-in-time)
train_all_mid = df.slice(0, final_oos_idx)['mid_price'].to_numpy()
final_daily_vol = np.diff(np.log(train_all_mid)).std() * np.sqrt(10 * 3600 * 24)

final_metrics, final_returns = run_backtest_on_window(
    df_final_oos, final_daily_vol, 'Final OOS'
)

print(f'\n  Sharpe:          {final_metrics["sharpe_ratio"]:.3f}')
print(f'  Sortino:         {final_metrics["sortino_ratio"]:.3f}')
print(f'  Max Drawdown:    {final_metrics["max_drawdown"]:.2%}')
print(f'  30s Accuracy:    {final_metrics["directional_acc_30s"]*100:.2f}%')
print(f'  Break-Even bps:  {final_metrics["break_even_bps"]:.2f}')
print(f'  Total Trades:    {final_metrics["total_trades"]}')

print('\n✅ NOTEBOOK 05 COMPLETE')
print('   Next step → Run 06_export_and_deploy.ipynb')